# Chấm điểm động tác chống đẩy — Bước 3: tạo tập dữ liệu rep

Đầu vào:
- thư mục keypoint của bước 1 (`keypoints/*.npz`, `pushup_index.csv`)
- `pushup_reps.csv` của bước 2

Đầu ra: một file `pushup_dataset.npz` chứa mọi rep đã cắt, chuẩn hoá, đưa về cùng độ dài và chia fold theo người tập.
Các notebook train (LSTM, ST-GCN) chỉ cần đọc file này.

## 1. Tìm dữ liệu đầu vào

In [ ]:
import os, glob

def find_one(name):
    found = glob.glob(f"/kaggle/working/**/{name}", recursive=True) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert found, f"Không thấy {name}. Hãy Add Input dữ liệu/notebook chứa file này."
    return found[0]

KP_ROOT = os.path.dirname(find_one("pushup_index.csv"))
REPS_CSV = find_one("pushup_reps.csv")
OUT_DIR = "/kaggle/working/pushup_dataset"
os.makedirs(OUT_DIR, exist_ok=True)
print("Keypoint:", KP_ROOT)
print("Reps    :", REPS_CSV)

## 2. Cấu hình

- **12 khớp**: vai, khuỷu, cổ tay, hông, gối, cổ chân (trái/phải). Bỏ điểm mặt (mặt bị làm mờ trong dataset) và các điểm bàn tay, bàn chân (nhiễu).
- **T = 32 khung** cho mỗi rep (một rep dài trung vị khoảng 23 khung ở 15 fps).
- Nhãn: `0 = good (đúng)`, `1 = bad (sai)`.
- Góc quay: `0 = front`, `1 = side`, `2 = diagonal`.

In [ ]:
import numpy as np
import pandas as pd

T = 32
JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
JOINT_NAMES = ["L_shoulder", "R_shoulder", "L_elbow", "R_elbow", "L_wrist", "R_wrist",
               "L_hip", "R_hip", "L_knee", "R_knee", "L_ankle", "R_ankle"]
# Cạnh của đồ thị khung xương (chỉ số theo JOINTS), dùng cho ST-GCN và để vẽ
EDGES = [(0, 1), (0, 2), (2, 4), (1, 3), (3, 5), (0, 6), (1, 7), (6, 7), (6, 8), (8, 10), (7, 9), (9, 11)]
SH, HIP = [0, 1], [6, 7]
VIEW_ID = {"front": 0, "side": 1, "diagonal": 2}
LABEL_ID = {"good": 0, "bad": 1}

## 3. Hàm xử lý

1. **Về pixel**: nhân x, y với chiều rộng, chiều cao video (video có nhiều kích thước khác nhau).
2. **Nội suy** các khung không phát hiện được người; visibility của các khung đó đặt bằng 0.
3. **Cắt** đoạn `[start, end]` của rep.
4. **Chuẩn hoá** theo từng rep:
   - Gốc toạ độ: vị trí trung vị của trung điểm hai hông trong rep.
   - Tỉ lệ: số lớn hơn giữa chiều dài thân và độ rộng vai (trung vị trong rep). Ở góc ngang hai vai chồng nhau nên dùng chiều dài thân;
     ở góc trước thân bị co ngắn do phối cảnh nên dùng độ rộng vai.
   - Gốc và tỉ lệ cố định trong cả rep, nên chuyển động lên/xuống của cơ thể được giữ nguyên.
   - Toạ độ sau chuẩn hoá bị chặn trong ±3 để loại các khớp bị đoán văng ra xa.
5. **Resample** về T khung bằng nội suy tuyến tính theo thời gian.

Toạ độ 3D (`pose_world_landmarks`) vốn đã tính bằng mét và có gốc tại hông nên chỉ cần cắt và resample.

In [ ]:
def load_video(file):
    d = np.load(os.path.join(KP_ROOT, "keypoints", file))
    det = d["detected"]
    xy = d["kp2d"][:, JOINTS, :2] * np.array([int(d["width"]), int(d["height"])], dtype=np.float32)
    vis = d["kp2d"][:, JOINTS, 3].copy()
    xyz = d["kp3d"][:, JOINTS, :3].copy()

    idx = np.arange(len(det))
    for arr in (xy, xyz):
        flat = arr.reshape(len(det), -1)          # view: sửa flat là sửa arr
        for k in range(flat.shape[1]):
            flat[~det, k] = np.interp(idx[~det], idx[det], flat[det, k])
    vis[~det] = 0.0
    return xy, xyz, vis


def resample(seq, n):
    pos = np.linspace(0, len(seq) - 1, n)
    lo = np.floor(pos).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    w = (pos - lo).reshape(-1, *[1] * (seq.ndim - 1))
    return seq[lo] * (1 - w) + seq[hi] * w


def normalize_2d(xy):
    mid_sh, mid_hip = xy[:, SH].mean(1), xy[:, HIP].mean(1)
    origin = np.median(mid_hip, axis=0)
    torso = np.median(np.linalg.norm(mid_sh - mid_hip, axis=-1))
    sh_width = np.median(np.linalg.norm(xy[:, SH[0]] - xy[:, SH[1]], axis=-1))
    out = (xy - origin) / (max(torso, sh_width) + 1e-6)
    return np.clip(out, -3, 3)      # chặn các khớp bị MediaPipe đoán văng ra xa (99% giá trị nằm trong ±1.8)


def make_sample(xy, xyz, vis, start, end):
    sl = slice(start, end + 1)
    v = resample(vis[sl], T)[..., None]
    x2d = np.concatenate([resample(normalize_2d(xy[sl]), T), v], axis=-1)     # (T, 12, 3): x, y, visibility
    x3d = np.concatenate([resample(xyz[sl], T), v], axis=-1)                  # (T, 12, 4): x, y, z, visibility
    return x2d.astype(np.float32), x3d.astype(np.float32)

## 4. Tạo tập dữ liệu

Chỉ lấy các rep có `ok = True`.

In [ ]:
reps = pd.read_csv(REPS_CSV)
reps = reps[reps.ok].sort_values(["file", "rep_id"]).reset_index(drop=True)   # các rep của một video nằm liền nhau

X2d, X3d = [], []
for file, group in reps.groupby("file", sort=False):
    xy, xyz, vis = load_video(file)
    for r in group.itertuples():
        a, b = make_sample(xy, xyz, vis, r.start, r.end)
        X2d.append(a)
        X3d.append(b)

X2d, X3d = np.stack(X2d), np.stack(X3d)
y = reps.quality.map(LABEL_ID).values.astype(np.int64)
view = reps["view"].map(VIEW_ID).values.astype(np.int64)
subject = reps.subject.values.astype(np.int64)

print("X2d:", X2d.shape, "| X3d:", X3d.shape, "| y:", y.shape)
print("Có NaN:", bool(np.isnan(X2d).any() or np.isnan(X3d).any()))

## 5. Chia fold theo người tập

5 fold, mỗi người chỉ nằm trong đúng một fold (`StratifiedGroupKFold`), cố gắng giữ tỉ lệ good/bad giữa các fold.
Khi train: fold `k` làm test, fold `(k+1) % 5` làm validation, 3 fold còn lại làm train.
Không bao giờ để rep của cùng một người vừa ở train vừa ở test.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

fold = np.zeros(len(y), dtype=np.int64)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for k, (_, test_idx) in enumerate(splitter.split(X2d, y, groups=subject)):
    fold[test_idx] = k

summary = pd.DataFrame({"fold": fold, "subject": subject, "y": y})
table = summary.groupby("fold").agg(so_nguoi=("subject", "nunique"), so_rep=("y", "size"), ti_le_bad=("y", "mean")).round(2)
table["nguoi"] = summary.groupby("fold").subject.apply(lambda s: sorted(s.unique()))
display(table)

## 6. Lưu

In [ ]:
np.savez_compressed(
    os.path.join(OUT_DIR, "pushup_dataset.npz"),
    X2d=X2d, X3d=X3d, y=y, view=view, subject=subject, fold=fold,
    rep_id=reps.rep_id.values, video=np.array(reps.file.tolist()),
    joint_names=np.array(JOINT_NAMES), edges=np.array(EDGES),
)
reps.assign(label=y, view_id=view, fold=fold).to_csv(os.path.join(OUT_DIR, "pushup_dataset_meta.csv"), index=False)

print("Số rep theo góc quay × nhãn:")
display(pd.crosstab(reps["view"], reps.quality))
print("Số rep theo người tập:")
display(pd.crosstab(reps.subject, reps.quality).T)

## 7. Kiểm tra bằng mắt

Mỗi hàng là một rep sau khi chuẩn hoá và resample, vẽ ở 6 thời điểm (đầu → đáy → cuối).
Xương màu đỏ: visibility thấp.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
times = np.linspace(0, T - 1, 6).astype(int)
fig, axes = plt.subplots(6, 6, figsize=(18, 16))
row = 0
for view_name, v in VIEW_ID.items():
    for label_name, lab in LABEL_ID.items():
        i = rng.choice(np.where((view == v) & (y == lab))[0])
        lim = np.abs(X2d[i, :, :, :2]).max() * 1.1
        for col, t in enumerate(times):
            ax = axes[row, col]
            p, vis_t = X2d[i, t, :, :2], X2d[i, t, :, 2]
            for a, b in EDGES:
                ax.plot([p[a, 0], p[b, 0]], [p[a, 1], p[b, 1]], lw=2,
                        color="tab:green" if min(vis_t[a], vis_t[b]) > 0.5 else "tab:red")
            ax.set_xlim(-lim, lim); ax.set_ylim(lim, -lim)      # trục y của ảnh hướng xuống
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(f"{view_name} / {label_name}\nsubject {subject[i]}")
            ax.set_title(f"t = {t}")
        row += 1
plt.tight_layout()
plt.show()

## 8. Kết quả

Tải `pushup_dataset.npz` về và tạo Kaggle Dataset từ nó (hoặc Save Version). Các notebook train chỉ cần file này.

In [ ]:
!ls -lh "{OUT_DIR}"